# conv-padding-zero — worked example 1: Symmetric 1-D zero padding by slice assignment

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-padding-zero`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Zero padding allocates a larger zero-filled buffer and copies the original signal into the interior, leaving zeros at the edges. For a *symmetric* pad of width `p`, the same amount `p` is added before and after the spatial dimension. Building it by hand (not `F.pad`) makes the allocate-then-assign mechanics explicit.

## Worked solution

We want to pad a `(B, IC, W)` tensor by `p` zeros on each side, giving width `W + 2*p`.

1. **Read the shape.** `B, IC, W = x.shape` so we know how big the buffer must be.
2. **Allocate a zero buffer.** `x.new_zeros(B, IC, W + 2*p)` gives a tensor of the right size that *inherits* `x`'s dtype and device — important so the copy doesn't trigger a cast or a device transfer. Every entry starts at zero, which is exactly what the padded border should be.
3. **Copy the interior.** The original signal must land in columns `[p : p + W]`. Slice assignment `out[..., p : p + W] = x` writes `x` into that window and leaves the first `p` and last `p` columns untouched (still zero).
4. **Why it works.** Because the buffer was zero-filled and we only overwrite the interior, the border is guaranteed zero. The interior is a verbatim copy of `x`, so no values are lost or altered.

In [ ]:
def pad1d_symmetric(x: Tensor, p: int) -> Tensor:
    B, IC, W = x.shape
    out = x.new_zeros(B, IC, W + 2 * p)
    out[..., p : p + W] = x
    return out

t.manual_seed(0)
x = t.arange(1, 13, dtype=t.float32).reshape(2, 1, 6)
y = pad1d_symmetric(x, 2)
print(y.shape)
print(y[0, 0])
print('border zero:', bool((y[..., :2] == 0).all()) and bool((y[..., -2:] == 0).all()))
print('interior matches:', bool((y[..., 2:8] == x).all()))